In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 22
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
15.827117800732776

Trial 1 =========================================
16.420222311772726

Trial 2 =========================================
14.034751502949575

Trial 3 =========================================
13.97698484681372

Trial 4 =========================================
14.365896462804187

Trial 5 =========================================
15.389828783230465

Trial 6 =========================================
13.239979372569227

Trial 7 =========================================
15.922600109373192

Trial 8 =========================================
13.417315981439957

Trial 9 =========================================
13.355405377780935

Trial 10 =========================================
14.840067594982452

Trial 11 =========================================
14.219829473174332

Trial 12 =========================================
13.71250970354965

Trial 13 =========================================
14.023086242139396

Trial 14 =========

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 18 =========================================
15.533942467350526

Trial 19 =========================================
13.189255912771745

Trial 20 =========================================
13.995307656667954

Trial 21 =========================================
14.452673866093729



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 22 =========================================
13.637150528381216

Trial 23 =========================================
13.612158657500922

Trial 24 =========================================
14.106784900796663

Trial 25 =========================================
15.316624310199012

Trial 26 =========================================
14.588710252277

Trial 27 =========================================
14.093718879738367

Trial 28 =========================================
17.354778433546073

Trial 29 =========================================
13.762070505278226

Trial 30 =========================================
13.815025490995582

Trial 31 =========================================
13.973986285063852

Trial 32 =========================================
15.872858327513942

Trial 33 =========================================
15.195330281486642

Trial 34 =========================================
15.771937534014835

Trial 35 =========================================
16.241130402229405

Trial 36 

...

Trial 39 =========================================
16.37221435311471

Trial 40 =========================================
13.800947333468574

Trial 41 =========================================
13.716201722966698

Trial 42 =========================================
14.26274294075356

Trial 43 =========================================
15.201468819234393

Trial 44 =========================================
14.956187786208536

Trial 45 =========================================
14.200111629652252

Trial 46 =========================================
14.019128005538374

Trial 47 =========================================
15.079930415899936

Trial 48 =========================================
15.252514704285758

Trial 49 =========================================
14.238396055288229

Trial 50 =========================================
13.987631956957628

Trial 51 =========================================
14.534309375994265

Trial 52 =========================================
16.198553454459606

Trial 53

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 56 =========================================
16.0999235624356

Trial 57 =========================================
15.164073418434352

Trial 58 =========================================
15.385441976202976

Trial 59 =========================================
14.138739111224714

Trial 60 =========================================
13.916564361796397

Trial 61 =========================================
14.022384409926993

Trial 62 =========================================
14.797497618588752

Trial 63 =========================================
17.42071341276224

Trial 64 =========================================
13.363663486397558

Trial 65 =========================================
14.978621780026863

Trial 66 =========================================
14.673335232750997

Trial 67 =========================================
13.277870658308304



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 68 =========================================
15.929222010399386

Trial 69 =========================================
15.994740660532326

Trial 70 =========================================
15.533942467350526

Trial 71 =========================================
14.34161210776282

Trial 72 =========================================
14.777637665377105

Trial 73 =========================================
15.282310562631453

Trial 74 =========================================
16.508750767866

Trial 75 =========================================
13.525138631012208

Trial 76 =========================================
13.425929776538274

Trial 77 =========================================
13.216774601810876

Trial 78 =========================================
13.907315247677817

Trial 79 =========================================
14.271702733748219

Trial 80 =========================================
14.030667796125005

Trial 81 =========================================
14.59575874196757

Trial 82 ==

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 17.76209332493813
Avg = 14.773729648117408
Std = 1.147148370131583


In [6]:
print(y_max_arr.tolist())

[15.827117800732776, 16.420222311772726, 14.034751502949575, 13.97698484681372, 14.365896462804187, 15.389828783230465, 13.239979372569227, 15.922600109373192, 13.417315981439957, 13.355405377780935, 14.840067594982452, 14.219829473174332, 13.71250970354965, 14.023086242139396, 15.856262952764313, 16.225453076078324, 13.142810120632978, 14.364733801667319, 15.533942467350526, 13.189255912771745, 13.995307656667954, 14.452673866093729, 13.637150528381216, 13.612158657500922, 14.106784900796663, 15.316624310199012, 14.588710252277, 14.093718879738367, 17.354778433546073, 13.762070505278226, 13.815025490995582, 13.973986285063852, 15.872858327513942, 15.195330281486642, 15.771937534014835, 16.241130402229405, 17.535492254189347, 15.826598114887712, 13.83857810337398, 16.37221435311471, 13.800947333468574, 13.716201722966698, 14.26274294075356, 15.201468819234393, 14.956187786208536, 14.200111629652252, 14.019128005538374, 15.079930415899936, 15.252514704285758, 14.238396055288229, 13.9876

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-22.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)